# Encoder Model Training with LoRA (PhayaThaiBERT)

This notebook fine-tunes **PhayaThaiBERT** (`clicknext/phayathaibert`) for Thai intent classification using **LoRA** (Low-Rank Adaptation) instead of full fine-tuning.

### Why LoRA instead of full fine-tuning?
- Full fine-tuning updates all 278M parameters — works but is expensive
- LoRA freezes the base model and injects small trainable adapter matrices
- Result: **~0.5% of parameters trained**, same or better accuracy, much less memory

### Pipeline overview
1. Install dependencies
2. Configure paths and parameters
3. Load and split dataset
4. Build label mappings
5. Load tokenizer and tokenize data
6. Load base model with classification head
7. Apply LoRA adapters
8. Configure training arguments
9. Set up MLflow lineage tracking
10. Train
11. Save the adapter

---
## 1. Install Dependencies

In [1]:
!pip install transformers==4.57.1 datasets==4.0.0 peft==0.17.0 trl==0.21.0 accelerate==1.10.0
!pip install "git+https://github.com/red-hat-data-services/mlflow@rhoai-3.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 196.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 662.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 316.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 655.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 689.9 MB/s  0:00:00
  Attempting uninstall: fsspec━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/14 [hf-xet]
    Found existing installation: fsspec 2026.1.0━━━━━━━━━━━━━━  3/14 [hf-xet]
    Uninstalling fsspec-2026.1.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/14 [hf-xet]
      Successfully uninstalled fsspec-2026.1.0━━━━━━━━━━━━━━━━  3/14 [hf-xet]
  Attempting uninstall: dill╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/14 [fsspec]
    Found existing installation: dill 0.3.9━━━━━━━━━━━━━━━━━━━  4/14 [fsspec]
    Uninstalling dill-0.3.9:m━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/14 [fsspec]
      Successfully uninstalled dill-0.3.9━━━━━━━━━━━━━━━━━━━━━━━━━  5/14 [dill]


---
## 2. Imports

| Library | Purpose |
|---|---|
| `transformers` | Model, tokenizer, Trainer, TrainingArguments, callbacks |
| `peft` | LoRA configuration — `LoraConfig`, `get_peft_model`, `TaskType` |
| `datasets` | Load JSONL dataset from disk or S3 |
| `mlflow` | Experiment tracking — log params, metrics, dataset lineage |
| `numpy` | `argmax` on logits to get predicted class |

In [2]:
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
import shutil
import os
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import numpy as np
import mlflow

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.7.1+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


### MLflow Authentication

MLflow tracks all training metrics, hyperparameters, and dataset lineage. Before using it, we need to set:
- **`MLFLOW_TRACKING_TOKEN`** — your OpenShift token 
- **`MLFLOW_TRACKING_URI`** — the MLflow server endpoint
- **`MLFLOW_WORKSPACE`** — the workspace where this experiment's runs are stored

In [3]:
os.environ["MLFLOW_TRACKING_TOKEN"] = os.environ.get("MLFLOW_TRACKING_TOKEN", "sha256~ifFt18KkwNyIzeBagf4AJ4oG-JZyai5_PO7vw2IdrgE")
os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
os.environ["MLFLOW_WORKSPACE"] = "encoder-sft"
os.environ["MLFLOW_RUN_NAME"] = "encoder-model-sftrun"
os.environ["MLFLOW_TRACKING_HEADERS"] = "{\"X-MLFLOW-WORKSPACE\": \"encoder-sft\", \"Content-Type\": \"application/json\"}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "encoder-model-finetuning"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
print("Environment variables set!")

Environment variables set!


---
## 3. Configuration

All paths and settings in one place. In the script version these come from `argparse`; here we set them directly.

| Parameter | Description |
|---|---|
| `MODEL_NAME` | Hugging Face model identifier for PhayaThaiBERT |
| `DATA_PATH` | Path to the JSONL training data (local or S3-downloaded) |
| `OUTPUT_DIR` | Where checkpoints and the final adapter are saved |
| `DATASET_SOURCE` | S3 URI or description for MLflow lineage tracking |
| `RUN_NAME` | Name that appears in the MLflow experiment dashboard |

In [4]:
MODEL_NAME = "clicknext/phayathaibert"
DATA_PATH = "/mnt/data/training_dataset.json"       # <-- change to your data path
OUTPUT_DIR = "/mnt/data/adapters"
DATASET_SOURCE = "/mnt/data/training_dataset.json"
RUN_NAME = "encoder-lora-run"

print(f"Model:      {MODEL_NAME}")
print(f"Data:       {DATA_PATH}")
print(f"Output:     {OUTPUT_DIR}")

Model:      clicknext/phayathaibert
Data:       /mnt/data/training_dataset.json
Output:     /mnt/data/adapters


---
## 4. Clean Output Directory

If a previous training run left files in `OUTPUT_DIR`, we remove them to avoid checkpoint corruption. Hugging Face Trainer can fail or produce wrong results when it finds stale checkpoints from a different configuration.

In [5]:
if os.path.exists(OUTPUT_DIR):
    print(f"Cleaning existing output dir: {OUTPUT_DIR}")
    shutil.rmtree(OUTPUT_DIR)
else:
    print(f"Output dir does not exist yet, will be created during training.")

Cleaning existing output dir: /mnt/data/adapters


---
## 5. Load Dataset

The dataset is a **JSONL** file where each line looks like:
```json
{"message": "บิลเดือนนี้ตัดวันไหนครับ", "intent": "MOBILE_BILLING_CHECK_DUE_DATE", "session_history": [...]}
```

`load_dataset("json", data_files=...)` reads all lines into a single Hugging Face `Dataset` object.

In [6]:
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

print(f"Total examples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"\nSample entry:")
dataset[0]

Total examples: 466
Columns: ['user_message', 'intent', 'session_history']

Sample entry:


{'user_message': 'งั้นช่วยเปิดไช้งานเบอร์รายเดือนนี้ให้เลยคับ ต้องการด่วนๆเลยพอดีพรุ่งจะไปต่างจังหวัดพอดีมีประชุมด่วนจจี๋',
 'intent': 'MOBILE_SIM_ACTIVATION_POSTPAID',
 'session_history': [{'content': 'สวัสดีค่ะ ติดต่อเรื่องเปิดใช้งานซิมรายเดือนใช่ไหมคะ',
   'role': 'assistant'},
  {'content': 'ใช่ครับ เพิ่งได้รับซิมใหม่มา ยังใช้ไม่ได้เลย เซง สุด แล่ว อ้อแล้วช่วยเช็ก  Network แถวบ้านให้หน่อยสัญญานไม่ดีเลย 📡 มีปัญหาตลอดโครหงุดหงิด',
   'role': 'user'},
  {'content': 'ได้ค่ะ รบกวนแจ้งหมายเลขที่ต้องการเปิดใช้งานก่อนนะคะ',
   'role': 'assistant'},
  {'content': 'เบอร์ 086-031-xxxx ครับ', 'role': 'user'},
  {'content': 'ตรวจสอบแล้วเป็นซิมรายเดือนที่รอเปิดใช้งานอยู่ค่ะ ต้องการให้ดำเนินการเปิดใช้งานเลยไหมครับ [option: {ทันที, รอการเปิดใช้งาน}]',
   'role': 'assistant'}]}

---
## 6. Build Label Mappings

The model outputs integer predictions (0, 1, 2, ...). We need a two-way mapping between intent names and these integers.

**Critical:** We sort the intents alphabetically so the mapping is deterministic and matches the evaluation script. Without sorting, the order depends on dataset row order which can change between runs.

In [7]:
unique_intents = sorted(dataset.unique("intent"))

num_labels = len(unique_intents)
label2id = {label: i for i, label in enumerate(unique_intents)}
id2label = {i: label for i, label in enumerate(unique_intents)}

print(f"Found {num_labels} unique intents:\n")
for label, idx in label2id.items():
    print(f"  {idx}: {label}")

Found 16 unique intents:

  0: MOBILE_BILLING_CHECK_DUE_DATE
  1: MOBILE_BILLING_PAYMENT_STATUS_POSTPAID
  2: MOBILE_BILLING_PAYMENT_STATUS_PREPAID
  3: MOBILE_NETWORK_CHECK_NO_SIGNAL
  4: MOBILE_PACKAGE_CHECK_DATA_CURRENT
  5: MOBILE_PACKAGE_CHECK_DATA_HISTORY
  6: MOBILE_ROAMING_CHECK_DATA_CURRENT
  7: MOBILE_ROAMING_CHECK_PAYMENT_POSTPAID
  8: MOBILE_SIM_ACTIVATION_POSTPAID
  9: MOBILE_SIM_ACTIVATION_PREPAID
  10: MOBILE_SIM_REPLACEMENT_POSPAID
  11: MOBILE_SIM_REPLACEMENT_PREPAID
  12: MOBILE_USAGE_CHECK_DATA_CURRENT
  13: MOBILE_USAGE_COMPARE_DATA_PLAN
  14: MOBILE_VAS_SUBSCRIBE_DATA_HISTORY
  15: MOBILE_VAS_SUBSCRIBE_DATA_SPECIFIC_MONTH


---
## 7. Split Dataset: Train / Validation / Test

We split the data into three parts:

| Split | % | Purpose |
|---|---|---|
| Train | 80% | Model learns from these examples |
| Validation | 10% | Evaluated after each epoch to detect overfitting |
| Test | 10% | Quarantined — never seen during training, used for final evaluation |

The two-step split: first 80/20, then the 20% is split 50/50 into val and test.

In [8]:
split_1 = dataset.train_test_split(test_size=0.2, seed=42)
split_2 = split_1["test"].train_test_split(test_size=0.5, seed=42)

train_dataset = split_1["train"]
val_dataset = split_2["train"]
test_dataset = split_2["test"]

print(f"Train:      {len(train_dataset)} examples")
print(f"Validation: {len(val_dataset)} examples")
print(f"Test:       {len(test_dataset)} examples")

Train:      372 examples
Validation: 47 examples
Test:       47 examples


### Save the quarantined test set

The test set is saved separately so it can be used later for final evaluation without any data leakage risk.

In [9]:
test_output_path = "/mnt/data/test_split.jsonl"
test_dataset.to_json(test_output_path)
print(f"Test set saved to {test_output_path}")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Test set saved to /mnt/data/test_split.jsonl


---
## 8. Load Tokenizer

The tokenizer converts Thai text into token IDs that the model understands. We **must** use the exact tokenizer that PhayaThaiBERT was pre-trained with — a mismatched tokenizer produces garbage because token ID `5698` means a different subword in a different vocabulary.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Max length: {tokenizer.model_max_length}")

sample = "บิลเดือนนี้ตัดวันไหนครับ"
print(f"\nSample: {sample}")
print(f"Tokens: {tokenizer.tokenize(sample)}")
print(f"IDs:    {tokenizer.encode(sample)}")

Vocab size: 248427
Max length: 510

Sample: บิลเดือนนี้ตัดวันไหนครับ
Tokens: ['▁', 'บิล', 'เดือนนี้', 'ตัด', 'วันไหน', 'ครับ']
IDs:    [5, 10, 4329, 11253, 600, 6262, 73, 6]


---
## 9. Tokenization Function

This function does two things for each example:

1. **Flattens session history + current message** into a single string:
   ```
   "Bot: สวัสดีครับ | User: ขอเช็กบิล | User: บิลเดือนนี้ตัดวันไหนครับ"
   ```
   This gives the model conversational context, not just the last message.

2. **Tokenizes** the flattened text with `padding="max_length"` and `truncation=True` at 512 tokens.

3. **Converts intent labels** to integer IDs using `label2id`.

The function is designed for **batched** processing — it receives lists of examples, not single examples.

In [11]:
def process_and_tokenize(examples):
    flattened_texts = []
    for i in range(len(examples["user_message"])):
        user_msg = examples["user_message"][i]
        session_history = examples.get("session_history", [[]] * len(examples["user_message"]))[i]

        history_texts = []
        for turn in (session_history or []):
            prefix = "Assistant: " if turn.get("role") == "assistant" else "User: "
            history_texts.append(prefix + turn.get("content", ""))

        current_msg = "User: " + user_msg
        flattened_texts.append(" | ".join(history_texts + [current_msg]))

    tokenized = tokenizer(flattened_texts, padding="max_length", truncation=True, max_length=512)
    tokenized["labels"] = [label2id[intent] for intent in examples["intent"]]
    return tokenized

### Apply tokenization to train and validation sets

`.map(batched=True)` processes the dataset in batches for speed. After this, each example has `input_ids`, `attention_mask`, and `labels` — everything the model needs.

In [12]:
print("Tokenizing train dataset...")
train_dataset = train_dataset.map(process_and_tokenize, batched=True)

print("Tokenizing validation dataset...")
val_dataset = val_dataset.map(process_and_tokenize, batched=True)

print(f"\nTrain columns: {train_dataset.column_names}")
print(f"Sample input_ids length: {len(train_dataset[0]['input_ids'])}")

Tokenizing train dataset...
Tokenizing validation dataset...


Map:   0%|          | 0/47 [00:00<?, ? examples/s]


Train columns: ['user_message', 'intent', 'session_history', 'input_ids', 'attention_mask', 'labels']
Sample input_ids length: 512


---
## 10. Load Base Model

`AutoModelForSequenceClassification` loads PhayaThaiBERT and adds a **classification head** on top:

```
Input tokens → [BERT: 12 layers × 768 hidden] → [CLS] → [Linear: 768 → num_labels] → logits
                ↑ pre-trained (knows Thai)                  ↑ new (random init)
```

We pass `label2id`/`id2label` so they are stored inside `config.json` — anyone loading the saved model later automatically knows the label mapping.

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    device_map="auto",
    trust_remote_code=True
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Base model parameters: {total_params:,}")
print(f"Classification head shape: {model.classifier.out_proj.weight.shape}")
print(f"  Maps {model.classifier.out_proj.in_features} hidden dims → {num_labels} labels")

Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at clicknext/phayathaibert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base model parameters: 277,486,096
Classification head shape: torch.Size([16, 768])
  Maps 768 hidden dims → 16 labels


---
## 11. Apply LoRA Adapters

Instead of updating all 278M parameters, LoRA injects small trainable matrices into the **attention layers** only:

```
Original:   W (768×768) + delta_W (768×768)     = 589,824 trainable params per layer
LoRA:       W (frozen)  + A(768×16) × B(16×768) =  24,576 trainable params per layer
```

### LoRA Configuration

| Parameter | Value | Why |
|---|---|---|
| `task_type` | `SEQ_CLS` | Sequence classification task |
| `r` | 16 | Rank of the low-rank matrices — higher = more capacity |
| `lora_alpha` | 32 (2×r) | Scaling factor — controls adapter contribution strength |
| `lora_dropout` | 0.05 | Small dropout on LoRA layers for regularization |
| `bias` | "none" | Don't train bias terms |
| `target_modules` | ["query", "value"] | Apply LoRA to Q and V attention matrices |
| `modules_to_save` | ["classifier"] | **Critical** — also save the classification head so it doesn't revert to random weights when reloading |

In [14]:
r = 16

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=r,
    lora_alpha=(2 * r),
    lora_dropout=0.05,
    bias="none",
    target_modules=["query", "value"],
    modules_to_save=["classifier"]
)

print(f"LoRA rank (r):      {peft_config.r}")
print(f"LoRA alpha:         {peft_config.lora_alpha}")
print(f"Target modules:     {peft_config.target_modules}")
print(f"Modules to save:    {peft_config.modules_to_save}")

LoRA rank (r):      16
LoRA alpha:         32
Target modules:     {'query', 'value'}
Modules to save:    ['classifier']


### Wrap the model with LoRA

`get_peft_model()` freezes all base model weights and injects LoRA adapter layers into the target modules. After this call, only the LoRA matrices and the classification head are trainable.

In [15]:
model = get_peft_model(model, peft_config)

model.print_trainable_parameters()

trainable params: 1,192,720 || all params: 278,678,816 || trainable%: 0.4280


---
## 12. Define Evaluation Metric

The Trainer calls `compute_metrics()` after each evaluation epoch. It receives raw logits and true labels, and must return a dictionary of metric scores.

Steps:
1. `np.argmax(predictions, axis=1)` — pick the class with the highest logit for each example
2. Compare with true labels to compute accuracy

In [16]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

---
## 13. Training Arguments

Every hyperparameter for the training loop is configured here:

| Parameter | Value | Purpose |
|---|---|---|
| `per_device_train_batch_size` | 16 | Examples per GPU per step |
| `learning_rate` | 2e-4 | Higher than full fine-tuning (2e-5) because LoRA adapters are small and start from zero |
| `num_train_epochs` | 5 | Passes through training data |
| `max_grad_norm` | 0.3 | Clip gradients to prevent exploding updates |
| `warmup_ratio` | 0.1 | First 10% of steps: LR ramps from 0 to target |
| `lr_scheduler_type` | cosine | After warmup, LR follows a cosine curve down to 0 |
| `fp16` | True | Half-precision for ~2x speed on GPU |
| `report_to` | mlflow | Log metrics to MLflow dashboard |
| `eval_strategy` | epoch | Evaluate on validation set after each epoch |
| `load_best_model_at_end` | True | After training, revert to the checkpoint with best accuracy |
| `save_total_limit` | 2 | Keep only the 2 most recent checkpoints to save disk |

In [17]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    logging_steps=25,
    num_train_epochs=5,
    max_grad_norm=0.3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    push_to_hub=False,
    report_to="mlflow",
    run_name=RUN_NAME,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    per_device_eval_batch_size=16,
)

print(f"Batch size:    {training_args.per_device_train_batch_size}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Epochs:        {training_args.num_train_epochs}")
print(f"FP16:          {training_args.fp16}")
print(f"Scheduler:     {training_args.lr_scheduler_type}")

Batch size:    16
Learning rate: 0.0002
Epochs:        5
FP16:          True
Scheduler:     SchedulerType.COSINE


---
## 14. MLflow Lineage Callback

Hugging Face Trainer has built-in MLflow integration that logs loss and metrics automatically. But it **does not** log:
- Where the training data came from (S3 path)
- LoRA-specific hyperparameters (rank, alpha, dropout)
- Which exact data splits were used

This custom callback injects that information at the start of training. It uses:
- `mlflow.log_params()` — log model name, data path, LoRA config
- `mlflow.data.from_pandas()` — register each data split as an MLflow dataset
- `mlflow.log_input()` — link datasets to the run with context labels (training/evaluation/test)

In [18]:
print(num_labels)

16


In [19]:
class MLflowLineageCallback(TrainerCallback):
    def __init__(self, model_name, data_path, dataset_source, peft_config, train_ds, val_ds, test_ds):
        self.model_name = model_name
        self.data_path = data_path
        self.dataset_source = dataset_source
        self.peft_config = peft_config
        self.train_ds = train_ds
        self.val_ds = val_ds
        self.test_ds = test_ds

    def on_train_begin(self, args, state, control, **kwargs):
        if state.is_world_process_zero:
            print("Injecting S3 data lineage into MLflow run...")

            mlflow.log_params({
                "model_name": self.model_name,
                "data_path": self.data_path,
                "lora_r": self.peft_config.r,
                "lora_alpha": self.peft_config.lora_alpha,
                "lora_dropout": self.peft_config.lora_dropout,
            })

            ds_train = mlflow.data.from_pandas(
                self.train_ds.select_columns(["user_message", "intent"]).to_pandas(),
                source=self.dataset_source, name="intent_train"
            )
            ds_val = mlflow.data.from_pandas(
                self.val_ds.select_columns(["user_message", "intent"]).to_pandas(),
                source=self.dataset_source, name="intent_val"
            )
            ds_test = mlflow.data.from_pandas(
                self.test_ds.select_columns(["user_message", "intent"]).to_pandas(),
                source=self.dataset_source, name="intent_test"
            )

            mlflow.log_input(ds_train, context="training")
            mlflow.log_input(ds_val, context="evaluation")
            mlflow.log_input(ds_test, context="test")

### Create the callback instance

In [20]:
lineage_callback = MLflowLineageCallback(
    model_name=MODEL_NAME,
    data_path=DATA_PATH,
    dataset_source=DATASET_SOURCE,
    peft_config=peft_config,
    train_ds=train_dataset,
    val_ds=val_dataset,
    test_ds=test_dataset
)

print("MLflow lineage callback ready.")

MLflow lineage callback ready.


---
## 15. Create Trainer

The `Trainer` combines everything into a single training loop:

| Argument | What it provides |
|---|---|
| `model` | The LoRA-wrapped model (frozen base + trainable adapters) |
| `args` | All hyperparameters from `TrainingArguments` |
| `train_dataset` | Tokenized training examples |
| `eval_dataset` | Tokenized validation examples |
| `tokenizer` | Needed for proper padding during evaluation |
| `compute_metrics` | Called after each eval to compute accuracy |
| `callbacks` | Our MLflow lineage callback |

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[lineage_callback]
)

print("Trainer ready.")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Trainer ready.


---
## 16. Train

This starts the training loop. For each of the 5 epochs:
1. Forward pass through all training batches → compute loss
2. Backward pass → compute gradients (only for LoRA + classifier params)
3. Optimizer step → update weights
4. Evaluate on validation set → log accuracy
5. Save checkpoint if this is the best accuracy so far

Loss and accuracy are logged every 25 steps to MLflow.

In [22]:
print("Starting training...")
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 6, 'bos_token_id': 5}.


Starting training...


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

Injecting S3 data lineage into MLflow run...


/opt/app-root/lib64/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for '/mnt/data/training_dataset.json'. Exception: 
  return _dataset_source_registry.resolve(
/opt/app-root/lib64/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/opt/app-root/lib64/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for '/mnt/data/training_dataset.json'. Exception: 
  return _dataset_source_registry.resolve(
/opt/app-root/lib64/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The spec

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.114819,0.510638
2,2.598100,1.151977,0.723404
3,1.617800,0.765224,0.851064
4,0.846500,0.651080,0.872340
5,0.571100,0.631511,0.872340


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

🏃 View run encoder-lora-run at: https://mlflow.redhat-ods-applications.svc.cluster.local:8443/#/workspaces/encoder-sft/experiments/10/runs/121ac70226c54a66ad750fc6bb01969d
🧪 View experiment at: https://mlflow.redhat-ods-applications.svc.cluster.local:8443/#/workspaces/encoder-sft/experiments/10


/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


TrainOutput(global_step=120, training_loss=1.263158098856608, metrics={'train_runtime': 40.2361, 'train_samples_per_second': 46.227, 'train_steps_per_second': 2.982, 'total_flos': 496263185694720.0, 'train_loss': 1.263158098856608, 'epoch': 5.0})

---
## 17. Save the LoRA Adapter

`trainer.save_model()` saves **only the LoRA adapter weights and the classification head** — not the full 278M-parameter base model. The saved files are small (~2-5 MB) because only the trained deltas are stored.

To use the model later:
```python
from peft import PeftModel
base = AutoModelForSequenceClassification.from_pretrained("clicknext/phayathaibert", ...)
model = PeftModel.from_pretrained(base, "/mnt/data/adapters/latest")
```

The base model is downloaded from Hugging Face Hub, and the small adapter is loaded on top.

In [23]:
final_output_dir = f"{OUTPUT_DIR}/latest"

print(f"Saving adapter to {final_output_dir}...")
trainer.save_model(final_output_dir)

print(f"\nSaved files:")
for f in sorted(os.listdir(final_output_dir)):
    size = os.path.getsize(os.path.join(final_output_dir, f))
    if size > 1e6:
        print(f"  {f:<40} {size/1e6:.1f} MB")
    else:
        print(f"  {f:<40} {size/1e3:.1f} KB")

print("\nTraining complete!")

Saving adapter to /mnt/data/adapters/latest...

Saved files:
  README.md                                5.2 KB
  adapter_config.json                      0.9 KB
  adapter_model.safetensors                4.8 MB
  special_tokens_map.json                  1.0 KB
  tokenizer.json                           17.3 MB
  tokenizer_config.json                    144.5 KB
  training_args.bin                        5.8 KB

Training complete!
